# Step 8e — DeBERTa fine-tune baseline

Auto-generated from the matching `.py` script. Run this notebook end-to-end on Colab.

**Setup**: mount Drive, set `PROJECT_ROOT`, install deps.


In [1]:
!pip install -q transformers accelerate

In [2]:
# Mount Drive + set PROJECT_ROOT
import os, sys
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/text-difficulty-classification'
except Exception:
    PROJECT_ROOT = os.path.abspath('.')
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
print('PROJECT_ROOT =', PROJECT_ROOT)


Mounted at /content/drive
PROJECT_ROOT = /content/drive/MyDrive/text-difficulty-classification


In [3]:
import argparse
import json
import os
import sys
import time

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

PROJECT_ROOT = os.environ['PROJECT_ROOT']
MULTI_DIR    = os.path.join(PROJECT_ROOT, 'outputs', 'multi_corpus')
MODELS_DIR   = os.path.join(PROJECT_ROOT, 'outputs', 'models_multi')
os.makedirs(MODELS_DIR, exist_ok=True)

LABEL_MAP = {'elementary': 0, 'middle': 1, 'high': 2}
RANDOM_SEED = 42


In [4]:
class TextDS(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts, self.labels = texts, labels
        self.tok, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, max_length=self.max_len,
                       padding='max_length', return_tensors='pt')
        return {'input_ids': enc['input_ids'][0],
                'attention_mask': enc['attention_mask'][0],
                'labels': torch.tensor(self.labels[i], dtype=torch.long)}


In [5]:
def _eval(y_true, y_pred):
    return {
        'f1_macro': float(f1_score(y_true, y_pred, average='macro',
                                   labels=[0, 1, 2], zero_division=0)),
        'f1_per_class': f1_score(y_true, y_pred, average=None,
                                 labels=[0, 1, 2], zero_division=0).tolist(),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'n': int(len(y_true)),
    }


In [6]:
def _list_ood():
    return sorted(f[len('ood_'):-len('.csv')]
                  for f in os.listdir(MULTI_DIR)
                  if f.startswith('ood_') and f.endswith('.csv'))


In [7]:
def _load_split(split):
    df = pd.read_csv(os.path.join(MULTI_DIR, f'{split}.csv'))
    bad = df[~df['education_level'].isin(LABEL_MAP)]
    if len(bad):
        sys.exit(f"[step8e] {split}: unknown labels: "
                 f"{bad['education_level'].dropna().unique().tolist()[:10]}")
    if df['full_text'].isna().any():
        sys.exit(f"[step8e] {split}: full_text has missing values")
    return (df['full_text'].astype(str).tolist(),
            df['education_level'].map(LABEL_MAP).astype(np.int64).to_numpy())


def _counts(y):
    return np.bincount(np.asarray(y, dtype=np.int64), minlength=3).tolist()


def _print_split_counts(name, y):
    print(f"[step8e] {name} label counts [elem, middle, high] = {_counts(y)}")


def _assert_finite_model(model, where):
    bad = []
    for name, param in model.named_parameters():
        if not torch.isfinite(param).all():
            bad.append(name)
            if len(bad) >= 5:
                break
    if bad:
        raise RuntimeError(f"[step8e] non-finite model weights {where}: {bad}")


def _assert_finite_grads(model, where):
    bad = []
    for name, param in model.named_parameters():
        if param.grad is not None and not torch.isfinite(param.grad).all():
            bad.append(name)
            if len(bad) >= 5:
                break
    if bad:
        raise RuntimeError(f"[step8e] non-finite gradients {where}: {bad}")


In [8]:
def _append_result(rec):
    out = os.path.join(MODELS_DIR, 'results.json')
    existing = json.load(open(out)) if os.path.exists(out) else []
    existing = [r for r in existing
                if not (r.get('feature_set') == rec['feature_set']
                        and r.get('classifier') == rec['classifier'])]
    existing.append(rec)
    json.dump(existing, open(out, 'w'), indent=2)


In [9]:
@torch.no_grad()
def predict(model, ds, device, batch_size=32):
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    preds = []
    for batch in tqdm(loader, desc='predict', leave=False):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        logits = model(input_ids=ids, attention_mask=mask).logits
        if not torch.isfinite(logits).all():
            raise RuntimeError("[step8e] non-finite logits during prediction")
        preds.append(logits.argmax(-1).cpu().numpy())
    return np.concatenate(preds)


In [10]:
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model', default='microsoft/deberta-v3-base')
    ap.add_argument('--epochs', type=int, default=3)
    ap.add_argument('--lr', type=float, default=2e-5)
    ap.add_argument('--head-lr', type=float, default=1e-4,
                    help='LR for newly initialized pooler/classifier params.')
    ap.add_argument('--batch-size', type=int, default=16)
    ap.add_argument('--max-len', type=int, default=512)
    ap.add_argument('--clip-grad', type=float, default=1.0)
    if any(a.endswith('.json') or a.startswith('-f') for a in sys.argv[1:]):
        args = ap.parse_args([])
    else:
        args = ap.parse_args()

    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                               get_linear_schedule_with_warmup)
    from torch.optim import AdamW

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cpu':
        sys.exit("[step8e] needs GPU — DeBERTa fine-tune on CPU = days")

    torch.manual_seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.set_default_dtype(torch.float32)

    print(f"[step8e] loading {args.model}")
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    model = AutoModelForSequenceClassification.from_pretrained(
        args.model, num_labels=3,
        id2label={0: 'elementary', 1: 'middle', 2: 'high'},
        label2id={'elementary': 0, 'middle': 1, 'high': 2})
    # DeBERTa-v3 can go unstable if a previous notebook cell changed the default
    # dtype or if mixed precision leaks in. Keep this baseline in FP32.
    model = model.float().to(device)
    _assert_finite_model(model, "after load")

    Xtr, ytr = _load_split('train')
    Xv,  yv  = _load_split('val')
    Xte, yte = _load_split('test')
    print(f"[step8e] train={len(Xtr)} val={len(Xv)} test={len(Xte)}")
    _print_split_counts('train', ytr)
    _print_split_counts('val', yv)
    _print_split_counts('test', yte)

    train_ds = TextDS(Xtr, ytr, tokenizer, args.max_len)
    val_ds   = TextDS(Xv, yv, tokenizer, args.max_len)
    test_ds  = TextDS(Xte, yte, tokenizer, args.max_len)

    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True)
    base_params, head_params = [], []
    for name, param in model.named_parameters():
        if name.startswith('classifier.') or name.startswith('pooler.'):
            head_params.append(param)
        else:
            base_params.append(param)
    optim = AdamW([
        {'params': base_params, 'lr': args.lr},
        {'params': head_params, 'lr': args.head_lr},
    ], weight_decay=0.01, eps=1e-6)
    total_steps = len(train_loader) * args.epochs
    sched = get_linear_schedule_with_warmup(optim,
                                            num_warmup_steps=int(0.1 * total_steps),
                                            num_training_steps=total_steps)

    t0 = time.time()
    best_val_f1 = -1.0
    best_state = None
    for epoch in range(args.epochs):
        model.train()
        running_loss, n_seen = 0.0, 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'epoch {epoch+1}')):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            optim.zero_grad(set_to_none=True)
            logits = model(input_ids=ids, attention_mask=mask).logits
            if not torch.isfinite(logits).all():
                raise RuntimeError(f"[step8e] non-finite logits at epoch={epoch+1} step={step}")
            loss = F.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f"[step8e] non-finite loss at epoch={epoch+1} step={step}")
            loss.backward()
            _assert_finite_grads(model, f"epoch={epoch+1} step={step}")
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip_grad)
            optim.step()
            sched.step()
            _assert_finite_model(model, f"after epoch={epoch+1} step={step}")
            running_loss += loss.item() * labels.size(0)
            n_seen += labels.size(0)
        val_pred = predict(model, val_ds, device)
        val_f1 = f1_score(yv, val_pred, average='macro', labels=[0, 1, 2], zero_division=0)
        val_per_class = f1_score(yv, val_pred, average=None,
                                 labels=[0, 1, 2], zero_division=0).tolist()
        print(f"[step8e] epoch {epoch+1}: loss={running_loss / max(n_seen, 1):.4f} "
              f"val_f1_macro={val_f1:.3f} pred_counts={_counts(val_pred)} "
              f"per_class={[round(x, 3) for x in val_per_class]}")
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

    test_pred = predict(model, test_ds, device)
    test_metrics = _eval(yte, test_pred)

    ood_metrics = {}
    for corpus in _list_ood():
        Xo, yo = _load_split(f'ood_{corpus}')
        ood_pred = predict(model, TextDS(Xo, yo, tokenizer, args.max_len), device)
        ood_metrics[corpus] = _eval(yo, ood_pred)

    rec = {
        'feature_set':  args.model.split('/')[-1],
        'classifier':   'fine-tune',
        'best_params':  {'epochs': args.epochs, 'lr': args.lr,
                         'head_lr': args.head_lr, 'bs': args.batch_size,
                         'clip_grad': args.clip_grad},
        'val_f1_macro': float(best_val_f1),
        'test':         test_metrics,
        'ood':          ood_metrics,
        'fit_seconds':  round(time.time() - t0, 1),
    }
    _append_result(rec)
    ood_str = [f"{c}={m['f1_macro']:.2f}" for c, m in ood_metrics.items()]
    print(f"[step8e] {rec['feature_set']}__fine-tune val={best_val_f1:.3f} "
          f"test={test_metrics['f1_macro']:.3f} ood={ood_str}")


In [11]:
main()


[step8e] loading microsoft/deberta-v3-base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

...

predict:   0%|          | 0/18 [00:00<?, ?it/s]

predict:   0%|          | 0/47 [00:00<?, ?it/s]

predict:   0%|          | 0/47 [00:00<?, ?it/s]

predict:   0%|          | 0/47 [00:00<?, ?it/s]

[step8e] deberta-v3-base__fine-tune val=0.886 test=0.879 ood=['onestop=0.29', 'openbookqa=0.14', 'race-high=0.19', 'race-middle=0.13']
